In [9]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ParquetSchemaDrift")
    .master("local[*]")
    .getOrCreate()
)


In [10]:
base_path = Path("data/schema_drift_demo")
batch_a_path = str(base_path / "batch_a")
batch_b_path = str(base_path / "batch_b")

batch_a = spark.createDataFrame(
    [
        (101, 500.0),
        (102, 250.0),
    ],
    ["order_id", "amount"]
)

batch_b = spark.createDataFrame(
    [
        (103, 300.0, "INR"),
        (104, 450.0, "USD"),
    ],
    ["order_id", "amount", "currency"]
)

batch_a.write.mode("overwrite").parquet(batch_a_path)
batch_b.write.mode("overwrite").parquet(batch_b_path)

In [11]:
print("Batch A schema:")
spark.read.parquet(batch_a_path).printSchema()

print("Reading the directory without mergeSchema:")

df_default = spark.read.parquet(batch_a_path, batch_b_path)
df_default.printSchema()
df_default.show()

print("Reading the directory with mergeSchema:")
df_merged = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(batch_a_path, batch_b_path)
)

df_merged.printSchema()
df_merged.orderBy("order_id").show()



spark.stop()

Batch A schema:
root
 |-- order_id: long (nullable = true)
 |-- amount: double (nullable = true)

Reading the directory without mergeSchema:
root
 |-- order_id: long (nullable = true)
 |-- amount: double (nullable = true)

+--------+------+
|order_id|amount|
+--------+------+
|     103| 300.0|
|     104| 450.0|
|     101| 500.0|
|     102| 250.0|
+--------+------+

Reading the directory with mergeSchema:
root
 |-- order_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)

+--------+------+--------+
|order_id|amount|currency|
+--------+------+--------+
|     101| 500.0|    NULL|
|     102| 250.0|    NULL|
|     103| 300.0|     INR|
|     104| 450.0|     USD|
+--------+------+--------+

